# RQ3 — Validity

In [2]:
import os, sys, warnings

warnings.filterwarnings("ignore")

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "results")):
    ROOT = os.path.dirname(ROOT)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from experiments._analysis import (
    load_rq3_data,
    rq3_validity_table,
    rq3_validity_by_model,
)
from experiments._loader import load_all_results, success_only

df_human = load_rq3_data(ROOT)

# All models, all modalities — needed for by-modality and by-model breakdowns
df_vlm_all = load_all_results()
df_vlm_all = success_only(df_vlm_all)

# RQ3 remains ImageNet-only because the human validity study and GT extraction are ImageNet-specific.
IMAGE_NET_SCENES = ["single/solo", "single/multi", "multi"]
df_vlm_all = df_vlm_all[df_vlm_all["obj_category"].isin(IMAGE_NET_SCENES)].copy()

# For the scene table: qwen multimodal only (as before)
df_vlm_qwen_mm = df_vlm_all[
    (df_vlm_all["model"] == "qwen")
    & (df_vlm_all["modality"] == "multimodal")
    & (df_vlm_all["obj_category"].isin(IMAGE_NET_SCENES))
]

MODEL_LABEL = EL = {
    "qwen": "Qwen3-VL",
    "kimi": "Kimi-VL",
    "intern": "InternVL-3.5",
    "gemma": "Gemma3-4B",
    "deepseek": "Deepseek-VL2",
    "nemotron": "Nemotron3-ON",
}

print(f"Human: {df_human.filename.nunique()} images, {len(df_human)} sessions")
print(f"VLM (all): {len(df_vlm_all)} results across {df_vlm_all.model.nunique()} models")

Human: 182 images, 224 sessions
VLM (all): 1918 results across 6 models


## Validity: accept rate · human IoU · VLM IoU

In [3]:
t = rq3_validity_table(df_human, df_vlm_qwen_mm)
display(t)
print(t.to_latex(escape=False))

,N sessions,Accept rate,"Human IoU (bbox, mean $\pm$ std)","VLM IoU (post-attack, mean $\pm$ std)"
Scene,,,,
Isolated,74,78.4%,0.877 $\pm$ 0.097,0.054 $\pm$ 0.144
Clustered,77,76.6%,0.848 $\pm$ 0.153,0.057 $\pm$ 0.122
Mixed,73,98.6%,0.652 $\pm$ 0.238,0.230 $\pm$ 0.197


\begin{tabular}{lrlll}
\toprule
 & N sessions & Accept rate & Human IoU (bbox, mean $\pm$ std) & VLM IoU (post-attack, mean $\pm$ std) \\
Scene &  &  &  &  \\
\midrule
Isolated & 74 & 78.4% & 0.877 $\pm$ 0.097 & 0.054 $\pm$ 0.144 \\
Clustered & 77 & 76.6% & 0.848 $\pm$ 0.153 & 0.057 $\pm$ 0.122 \\
Mixed & 73 & 98.6% & 0.652 $\pm$ 0.238 & 0.230 $\pm$ 0.197 \\
\bottomrule
\end{tabular}



## Validity by model

In [4]:
t_model = rq3_validity_by_model(df_human, df_vlm_all, model_label=MODEL_LABEL)
display(t_model)
print(t_model.to_latex(escape=False))

,N (VLM),Accept rate,"Human IoU (bbox, mean $\pm$ std)","VLM IoU (post-attack, mean $\pm$ std)"
Model,,,,
Deepseek-VL2,13,84.4%,0.794 $\pm$ 0.197,0.019 $\pm$ 0.062
Gemma3-4B,3,84.4%,0.794 $\pm$ 0.197,0.091 $\pm$ 0.032
InternVL-3.5,378,84.4%,0.794 $\pm$ 0.197,0.131 $\pm$ 0.118
Kimi-VL,270,84.4%,0.794 $\pm$ 0.197,0.057 $\pm$ 0.141
Nemotron3-ON,447,84.4%,0.794 $\pm$ 0.197,0.202 $\pm$ 0.240
Qwen3-VL,807,84.4%,0.794 $\pm$ 0.197,0.256 $\pm$ 0.297


\begin{tabular}{lrlll}
\toprule
 & N (VLM) & Accept rate & Human IoU (bbox, mean $\pm$ std) & VLM IoU (post-attack, mean $\pm$ std) \\
Model &  &  &  &  \\
\midrule
Deepseek-VL2 & 13 & 84.4% & 0.794 $\pm$ 0.197 & 0.019 $\pm$ 0.062 \\
Gemma3-4B & 3 & 84.4% & 0.794 $\pm$ 0.197 & 0.091 $\pm$ 0.032 \\
InternVL-3.5 & 378 & 84.4% & 0.794 $\pm$ 0.197 & 0.131 $\pm$ 0.118 \\
Kimi-VL & 270 & 84.4% & 0.794 $\pm$ 0.197 & 0.057 $\pm$ 0.141 \\
Nemotron3-ON & 447 & 84.4% & 0.794 $\pm$ 0.197 & 0.202 $\pm$ 0.240 \\
Qwen3-VL & 807 & 84.4% & 0.794 $\pm$ 0.197 & 0.256 $\pm$ 0.297 \\
\bottomrule
\end{tabular}

